In [1]:
# Load libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# Load the data
companies = pd.read_csv('/Users/akhilkumarmarni/Downloads/cyndx_challenge/companies.csv')
companies.head()


,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City
0,Molan Steel Co.,11-50,Molan Steel Co. engages in supplying steel pro...,molansteel.com,RY,SA,Riyadh/Saudi Arabia Metro,Riyadh
1,Hunter Douglas NV,10001+,"Hunter Douglas NV engages in the design, manuf...",hunterdouglas.com,ZH,NL,Amsterdam/NL Metro,Rotterdam
2,"Root, Inc.",501-1000,"Root, Inc. is a technology insurance company, ...",inc.joinroot.com,OH,US,Columbus/OH Metro,Columbus
3,Bowlero Corp.,10001+,Bowlero Corp. engages in operating bowling cen...,bowlero.com,VA,US,Richmond/VA Metro,Mechanicsville
4,Meridia Real Estate III SOCIMI SA,1-10,Meridia Real Estate III SOCIMI SA operates as ...,meridiarealestateiiisocimi.com,CT,ES,Barcelona/Spain Metro,Barcelona


In [20]:
# Filter rows where the 'country' column is 'US'
us_rows = companies[companies['Country'] == 'US']

# Print the filtered rows
print(us_rows)

                                 Name EmployeeCount  \
2                          Root, Inc.      501-1000   
3                       Bowlero Corp.        10001+   
5                Lumos Networks Corp.      501-1000   
8         Delta Natural Gas Co., Inc.       101-250   
9      Medical Media Television, Inc.          1-10   
...                               ...           ...   
16727      Soligen Technologies, Inc.        51-100   
16728        CenterPoint Energy, Inc.        10001+   
16729              Glori Energy, Inc.         11-50   
16730         Bald Eagle Energy, Inc.          1-10   
16731                PGI Energy, Inc.       101-250   

                                             Description  \
2      Root, Inc. is a technology insurance company, ...   
3      Bowlero Corp. engages in operating bowling cen...   
5      Lumos Networks Corp. engages in the provision ...   
8      Delta Natural Gas Co., Inc. engages in the dis...   
9      Medical Media Television, Inc. e

In [23]:
print("Filtered data saved to 'usa_companies.csv'")

Filtered data saved to 'usa_companies.csv'


In [6]:
def search(query: str, companies: pd.DataFrame, n=100) -> pd.DataFrame:
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(companies['Description'].fillna(''))

    query_vec = vectorizer.transform([query])
    cosine_sim = cosine_similarity(query_vec, tfidf_matrix).flatten()

    companies['similarity'] = cosine_sim
    results = companies.sort_values(by='similarity', ascending=False).head(n)

    return results


In [10]:
def search(query: str, companies: pd.DataFrame, n=100) -> pd.DataFrame:
    # Combine relevant fields for search (you can adjust this!)
    corpus = companies['Description'].fillna('') + ' ' + companies['Name'].fillna('')
    
    # Vectorize using TF-IDF
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(corpus)
    
    # Vectorize query
    query_vec = vectorizer.transform([query])
    
    # Compute cosine similarity
    similarity = cosine_similarity(query_vec, X).flatten()
    
    # Add similarity to DataFrame
    companies['similarity'] = similarity
    
    # Return top n results sorted by similarity
    results = companies.sort_values(by='similarity', ascending=False).head(n).copy()
    
    return results


In [17]:
corpus = companies['Description'].fillna('') + ' ' + companies['Name'].fillna('')
v = TfidfVectorizer()
transformed_output = v.fit_transform(corpus)
print(v.vocabulary_)


{'molan': 31349, 'steel': 45096, 'co': 10145, 'engages': 15651, 'in': 23194, 'supplying': 45911, 'products': 37908, 'it': 24362, 'specializes': 44579, 'the': 47311, 'production': 37900, 'and': 2835, 'supply': 45910, 'of': 34117, 'high': 21698, 'quality': 38588, 'trade': 48179, 'saudi': 41816, 'arabia': 3348, 'abroad': 863, 'company': 10565, 'was': 51161, 'founded': 18231, 'on': 34370, 'august': 4202, '2015': 381, 'is': 24245, 'headquartered': 21236, 'riyadh': 40545, 'hunter': 22418, 'douglas': 14056, 'nv': 33854, 'design': 13092, 'manufacture': 29200, 'marketing': 29373, 'blinds': 6342, 'architectural': 3423, 'operates': 34520, 'through': 47511, 'following': 17990, 'segments': 42435, 'window': 51778, 'covering': 11550, 'segment': 42433, 'manufactures': 29204, 'coverings': 11551, 'for': 18049, 'commercial': 10501, 'residential': 40045, 'use': 49590, 'refers': 39523, 'to': 47792, 'sale': 41435, 'by': 7749, 'henry': 21505, 'sonnenberg': 44352, '1919': 268, 'rotterdam': 40941, 'netherlands

In [11]:
def rerank(results: pd.DataFrame, n=10, lambda_param=0.5) -> pd.DataFrame:
    # Vectorize using TF-IDF again on selected results
    corpus = results['Description'].fillna('') + ' ' + results['Name'].fillna('')
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(corpus)
    
    # Query vector (use first result as the best approximation)
    query_vec = vectorizer.transform([corpus.iloc[0]])
    
    # Precompute similarities
    sim_to_query = cosine_similarity(X, query_vec).flatten()
    sim_between_docs = cosine_similarity(X)
    
    # Initialize MMR selection
    selected_indices = []
    candidate_indices = list(range(X.shape[0]))
    
    while len(selected_indices) < n and candidate_indices:
        mmr_score = []
        for idx in candidate_indices:
            if not selected_indices:
                diversity_term = 0
            else:
                diversity_term = max(sim_between_docs[idx][j] for j in selected_indices)
            
            score = lambda_param * sim_to_query[idx] - (1 - lambda_param) * diversity_term
            mmr_score.append((idx, score))
        
        # Select doc with highest MMR score
        selected_idx = max(mmr_score, key=lambda x: x[1])[0]
        selected_indices.append(selected_idx)
        candidate_indices.remove(selected_idx)
    
    # Return reranked results
    reranked_results = results.iloc[selected_indices].reset_index(drop=True)
    
    return reranked_results


In [12]:
rerank(search('companies who manufacture steel products', companies))


,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City,similarity
0,"Yamato Kogyo Co., Ltd.",1001-5000,"Yamato Kogyo Co., Ltd. engages in the manageme...",yamatokogyo.co.jp,HY,JP,Osaka/Japan Metro,Himeji,0.578412
1,Argent Industrial Ltd.,1001-5000,Argent Industrial Ltd. operates as a holding c...,argent.co.za,NL,ZA,Johannesburg/S.Afr. Metro,Durban,0.498322
2,Egyptian Iron & Steel,10001+,Egyptian Iron & Steel engages in the manufactu...,hadisolb.com,HU,EG,Africa (Northern) Metro,Helwan,0.495925
3,"Steel Dynamics, Inc.",10001+,"Steel Dynamics, Inc. engages in the manufactur...",steeldynamics.com,IN,US,Indianapolis/IN Metro,Fort Wayne,0.491008
4,Kyoei Steel Ltd.,1001-5000,"Kyoei Steel Ltd. engages in the manufacture, s...",kyoeisteel.co.jp,OS,JP,Osaka/Japan Metro,Osaka,0.483062
5,Swiss Steel Holding AG,5001-10000,Swiss Steel Holding AG engages in the producti...,swisssteel-group.com,LU,CH,Zurich/Switzerland Metro,Lucerne,0.444777
6,Kori Holdings Ltd.,251-500,"Kori Holdings Ltd. is an investment company, w...",kori.com.sg,CE,SG,Singapore Metro,Singapore,0.408395
7,Asia Enterprises Holding Ltd.,51-100,Asia Enterprises Holding Ltd. engages in the d...,asiaenterprises.com.sg,CE,SG,Singapore Metro,Singapore,0.390037
8,"Sanyu Co., Ltd.",251-500,"Sanyu Co., Ltd. engages in the manufacture and...",sanyu-cfs.co.jp,OS,JP,Osaka/Japan Metro,Hirakata,0.369952
9,Saudi Steel Pipe Co.,251-500,Saudi Steel Pipe Co. engages in the manufactur...,sspipe.com,EP,SA,Riyadh/Saudi Arabia Metro,Dammam,0.345528


In [7]:
def rerank(results: pd.DataFrame, n=10, lambda_param=0.5) -> pd.DataFrame:
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(results['Description'].fillna(''))

    query_vec = vectorizer.transform([query])
    cosine_sim_to_query = cosine_similarity(query_vec, tfidf_matrix).flatten()

    selected_indices = []
    candidate_indices = list(range(len(results)))

    # First select the most relevant document
    first_idx = np.argmax(cosine_sim_to_query)
    selected_indices.append(first_idx)
    candidate_indices.remove(first_idx)

    while len(selected_indices) < n and candidate_indices:
        mmr_scores = []
        for idx in candidate_indices:
            sim_to_query = cosine_sim_to_query[idx]
            sim_to_selected = max([cosine_similarity(tfidf_matrix[idx], tfidf_matrix[j])[0][0] for j in selected_indices])
            mmr_score = lambda_param * sim_to_query - (1 - lambda_param) * sim_to_selected
            mmr_scores.append(mmr_score)

        next_idx = candidate_indices[np.argmax(mmr_scores)]
        selected_indices.append(next_idx)
        candidate_indices.remove(next_idx)

    reranked_results = results.iloc[selected_indices].reset_index(drop=True)
    return reranked_results


In [8]:
query = 'companies who manufacture steel products'
rerank(search(query, companies), n=10)


,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City,similarity
0,UnipolSai Assicurazioni SpA,10001+,UnipolSai Assicurazioni SpA engages in providi...,unipolsai.com,BO,IT,Milan/Italy Metro,Bologna,0.282150
1,Egyptian Iron & Steel,10001+,Egyptian Iron & Steel engages in the manufactu...,hadisolb.com,HU,EG,Africa (Northern) Metro,Helwan,0.500679
2,Abu Dhabi National Co. for Building Materials,11-50,Abu Dhabi National Co. for Building Materials ...,bildco.ae,AB,AE,Abu Dhabi/UAE Metro,Abu Dhabi,0.322926
3,Corporación Aceros Arequipa SA,1001-5000,Corporación Aceros Arequipa SA is engages in t...,acerosarequipa.com,LP,PE,South America Metro,Magdalena del Mar,0.303593
4,IMCD NV,1001-5000,"IMCD NV engages in the sale, marketing, and di...",imcdgroup.com,ZH,NL,Amsterdam/NL Metro,Rotterdam,0.210645
5,Georgia Capital Plc,11-50,"Georgia Capital Plc is a holding company, whic...",georgiacapital.ge,GL,GB,London/UK Metro,London,0.261664
6,One Group Corp.,501-1000,One Group Corp. engages in the management of i...,ogicgroup.co.jp,OS,JP,Osaka/Japan Metro,Osaka,0.248900
7,"Porch Group, Inc.",501-1000,"Porch Group, Inc. engages in the development a...",porchgroup.com,WA,US,Seattle/WA Metro,Seattle,0.216634
8,GPH Ispat Ltd.,1001-5000,GPH Ispat Ltd. engages in the manufacture of s...,gphispat.com.bd,CG,BD,Asia (South/West) Metro,Chattogram,0.408454
9,Salzgitter AG,10001+,Salzgitter AG engages in the manufacture of st...,salzgitter-ag.de,NI,DE,Hamburg/Germany Metro,Salzgitter,0.393016


In [9]:
rerank(search('companies who manufacture steel products', companies))


,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City,similarity
0,UnipolSai Assicurazioni SpA,10001+,UnipolSai Assicurazioni SpA engages in providi...,unipolsai.com,BO,IT,Milan/Italy Metro,Bologna,0.282150
1,Egyptian Iron & Steel,10001+,Egyptian Iron & Steel engages in the manufactu...,hadisolb.com,HU,EG,Africa (Northern) Metro,Helwan,0.500679
2,Abu Dhabi National Co. for Building Materials,11-50,Abu Dhabi National Co. for Building Materials ...,bildco.ae,AB,AE,Abu Dhabi/UAE Metro,Abu Dhabi,0.322926
3,Corporación Aceros Arequipa SA,1001-5000,Corporación Aceros Arequipa SA is engages in t...,acerosarequipa.com,LP,PE,South America Metro,Magdalena del Mar,0.303593
4,IMCD NV,1001-5000,"IMCD NV engages in the sale, marketing, and di...",imcdgroup.com,ZH,NL,Amsterdam/NL Metro,Rotterdam,0.210645
5,Georgia Capital Plc,11-50,"Georgia Capital Plc is a holding company, whic...",georgiacapital.ge,GL,GB,London/UK Metro,London,0.261664
6,One Group Corp.,501-1000,One Group Corp. engages in the management of i...,ogicgroup.co.jp,OS,JP,Osaka/Japan Metro,Osaka,0.248900
7,"Porch Group, Inc.",501-1000,"Porch Group, Inc. engages in the development a...",porchgroup.com,WA,US,Seattle/WA Metro,Seattle,0.216634
8,GPH Ispat Ltd.,1001-5000,GPH Ispat Ltd. engages in the manufacture of s...,gphispat.com.bd,CG,BD,Asia (South/West) Metro,Chattogram,0.408454
9,Salzgitter AG,10001+,Salzgitter AG engages in the manufacture of st...,salzgitter-ag.de,NI,DE,Hamburg/Germany Metro,Salzgitter,0.393016


In [3]:
def search(query: str, companies: pd.DataFrame, n=100) -> pd.DataFrame:
    # Basic string match on company name
    results = companies[companies['Name'].str.contains(query, case=False, na=False)].copy()
    results = results.head(n)
    return results


In [4]:
def rerank(results: pd.DataFrame, n=10, lambda_param=0.5) -> pd.DataFrame:
    # For this example, use 'Description' column if available, else 'Name'
    if 'Description' in results.columns:
        text_column = results['Description'].fillna(results['Name'])
    else:
        text_column = results['Name']
    
    # Vectorize the text
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(text_column.tolist())
    
    # Vectorize the query (for simplicity, using first entry as query proxy here)
    query_vector = tfidf_matrix.mean(axis=0)
    
    # Compute relevance scores: cosine similarity to query vector
    relevance = cosine_similarity(tfidf_matrix, query_vector).flatten()
    
    # Compute pairwise similarities between documents
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    # Initialize selected set
    selected_indices = []
    candidate_indices = list(range(len(results)))
    
    # Select first result (highest relevance)
    first_idx = relevance.argmax()
    selected_indices.append(first_idx)
    candidate_indices.remove(first_idx)
    
    # Iteratively select next best according to MMR
    while len(selected_indices) < n and candidate_indices:
        mmr_score = []
        
        for idx in candidate_indices:
            sim_to_query = relevance[idx]
            sim_to_selected = max([similarity_matrix[idx][j] for j in selected_indices])
            
            mmr = lambda_param * sim_to_query - (1 - lambda_param) * sim_to_selected
            mmr_score.append((idx, mmr))
        
        # Select item with highest MMR score
        next_idx = max(mmr_score, key=lambda x: x[1])[0]
        selected_indices.append(next_idx)
        candidate_indices.remove(next_idx)
    
    # Return the re-ranked DataFrame
    reranked_df = results.iloc[selected_indices].copy()
    reranked_df['MMR_Rank'] = range(1, len(selected_indices) + 1)
    
    return reranked_df.reset_index(drop=True)


In [5]:
rerank(search('companies who manufacture steel products', companies))


ValueError: empty vocabulary; perhaps the documents only contain stop words

In [18]:
# Filter rows where the 'country' column is 'US'
us_rows = df[df['country'] == 'US']

# Print the filtered rows
print(us_rows)

NameError: name 'df' is not defined